# 03 · LLM zero/few-shot (tarea 4 del flujo)

Un solo notebook para toda la tarea 4. Se ejecuta de arriba abajo en Kaggle y al final deja fijada **la mejor configuración** (modelo + prompt + nº de ejemplos) en `results/llm/mejor_config.json`, para que las tareas 5 (RAG) y 6 (calibración y evaluación) partan de ella.

| Fase | Qué hace | Salida en `results/llm/` |
|---|---|---|
| 4.1 Piloto | 4 LLMs abiertos en zero-shot sobre dev, mismo prompt y misma entrada limpia | `piloto_dev_<modelo>.csv`, `piloto_dev_resumen.csv` |
| 4.2 Prompt | Los 2 mejores del piloto con un prompt más detallado | `ajuste_dev_<modelo>__detallado_k0.csv` |
| 4.3 Few-shot | Esos 2 modelos con 4 y 8 ejemplos de train (mitad Fake, mitad Verdadera) | `ajuste_dev_<modelo>__<prompt>_k<k>.csv` |
| 4.4 Elección | Tabla con todo, IC del AUC por *bootstrap* y elección automática | `ajuste_dev_resumen.csv`, `mejor_config.json` |
| 4.5 Test | Puntúa test **una sola vez** con la configuración elegida (sin mirar métricas) | `test_<modelo>__<prompt>_k<k>.csv` |

- **Clasificación:** una sola pasada por noticia y se lee la probabilidad del primer token de la respuesta. `p_fake = P(Falsa) / (P(Falsa) + P(Verdadera))`. Así hay AUC y se pueden aplicar las mismas reglas de decisión que a los baselines (umbral 0,5 y rango al 50 %).
- **Criterio de elección:** mayor AUC en dev (no depende del umbral); desempate por F1 con umbral 0,5. Se descartan configuraciones con `p_formato` < 0,95 o con `nan`.
- **Test:** no se usa para decidir nada ni se calculan métricas con él aquí; la evaluación es de la tarea 6.
- **Reanudable:** cada CSV se guarda cada 25 noticias y lo que ya esté completo no se vuelve a ejecutar.

## Entorno

Pensado para **Kaggle**: *Accelerator* → GPU T4 ×2, *Internet* activado y un secreto `HF_TOKEN` (*Add-ons → Secrets*). Los Gemma son modelos restringidos: hay que aceptar su licencia en Hugging Face (inmediato) con la misma cuenta del token. Los Qwen no lo necesitan.

La celda clona el repo, instala dependencias y genera `data/processed` si no existe. En local (sin GPU) el notebook se ejecuta igual pero se salta los modelos.

Los resultados de cada modelo están versionados en `results/llm/`; si el CSV de un modelo ya está completo, ese modelo no se vuelve a ejecutar. Para el resumen no hace falta GPU.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
REPO, BRANCH = "https://github.com/Rodrigo-Romay/Practica-TAPLN.git", "feature/LLM"

if IN_KAGGLE:
    os.environ["HF_HOME"] = "/tmp/hf"   # más disco libre que /kaggle/working
    ROOT = Path("/kaggle/working/Practica-TAPLN")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers", "accelerate", "bitsandbytes"], check=True)
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Sin HF_TOKEN: solo funcionarán los modelos no restringidos (Qwen).", e)
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "data" / "processed" / "train.csv").exists():
    raw = ROOT / "data" / "raw"
    subprocess.run([sys.executable, str(ROOT / "src" / "process_data.py"),
                    "--train", str(raw / "train.xlsx"), "--dev", str(raw / "development.xlsx"),
                    "--test", str(raw / "test.xlsx"), "--out", str(ROOT / "data" / "processed")], check=True)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

import data_utils as du
import preprocessing as pp
import evaluation as ev
import baselines as bl
import llm

splits = du.load_clean_splits(ROOT)
media_re = pp.media_regex_from_splits(splits)   # nombres de medio de train/dev
dev_in = llm.llm_inputs(splits["dev"], media_re)
train_in = llm.llm_inputs(splits["train"], media_re)   # solo para sacar ejemplos few-shot
OUT = ROOT / "results" / "llm"
OUT.mkdir(parents=True, exist_ok=True)

print(f"dev: {len(dev_in)} noticias, {dev_in.label.mean():.1%} Fake")
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "ninguna")

## Prompt

Mismo prompt para todos los modelos en el piloto (en `src/llm.py`): rol de verificador, aviso de que se han quitado enlaces y medios y de que las cifras aparecen como `NUM`, titular («no disponible» si falta) y cuerpo cortado a 800 palabras.

El modo *thinking* de Qwen3 está desactivado y **no se puede probar con este método**: con *thinking* el primer token es el inicio del razonamiento y no la respuesta, así que `p_fake` no tendría sentido (`llm.run` lo rechaza). Probarlo exigiría generar el razonamiento y leer la respuesta después, con un coste de decenas de segundos por noticia; queda fuera de esta tarea.

En 4.2 se añade una variante `detallado` del prompt de sistema (criterios de verificación, sin atajos de superficie como mayúsculas o longitud, que el EDA descartó).

In [ ]:
print(llm.SYSTEM, "\n")
print(llm.USER.format(headline=dev_in.headline[0], body=dev_in.body[0][:300] + " [...]"))

## Candidatos

| Clave | Modelo | Carga en T4 | Licencia |
|---|---|---|---|
| `qwen3-8b` | Qwen/Qwen3-8B | fp16, repartido en 2×T4 | Apache 2.0 |
| `gemma3-12b` | google/gemma-3-12b-it | 4 bits (NF4) | Gemma Terms |
| `qwen38-27b` | Qwen/Qwen3.8-27B | 4 bits (NF4) | ver ficha del modelo |
| `gemma4-31b` | google/gemma-4-31B-it | 4 bits (NF4) | Gemma Terms |

Llama-3.1-8B-Instruct se consideró como candidato, pero su acceso en Hugging Face requiere aprobación manual de Meta (no inmediata); queda comentado por si se obtiene más adelante y **no entra en esta comparación**.

Los modelos de 12B o más van en 4 bits porque en fp16 no caben en los 32 GB disponibles (Gemma 3 12B ocupa ~24 GB). Si alguno da `nan` (desbordamiento de fp16), se anota y se repite con otra configuración.

Cada modelo guarda `results/llm/piloto_dev_<clave>.csv` y, si la sesión se corta, reanuda desde ahí. Con `EJECUTAR` se elige qué modelos lanzar; los que ya tienen el CSV completo se saltan.

In [ ]:
MODELOS = {
    "qwen3-8b":    {"name": "Qwen/Qwen3-8B"},
    # "llama31-8b": {"name": "meta-llama/Llama-3.1-8B-Instruct"},  # acceso pendiente de aprobación
    "gemma3-12b":  {"name": "google/gemma-3-12b-it", "quant": "4bit"},
    "qwen38-27b":  {"name": "Qwen/Qwen3.8-27B", "quant": "4bit"},
    "gemma4-31b":  {"name": "google/gemma-4-31B-it", "quant": "4bit"},
}
EJECUTAR = list(MODELOS)   # los que ya tienen su CSV completo en results/llm se saltan
BORRAR_CACHE = IN_KAGGLE   # borra cada modelo del disco al terminar con él

In [ ]:
import shutil
if IN_KAGGLE:
    shutil.rmtree("/tmp/hf", ignore_errors=True)   # borra lo que dejó Qwen y el intento anterior de Gemma

In [ ]:
import shutil

# Ajuste (4.2-4.3): modelos que pasan, ejemplos few-shot y semilla
TOP_N = 2                 # los TOP_N modelos con mayor AUC en el piloto pasan al ajuste
K_FEWSHOT = (4, 8)        # nº de ejemplos few-shot (pares: mitad Fake, mitad Verdadera)
SEED_EJ, MAX_WORDS_EJ = 42, 150
PUNTUAR_TEST = True       # 4.5: puntuar test con la configuración elegida (sin mirar métricas)


def dev_path(key, prompt="base", k=0):
    """CSV de una configuración en dev. El zero-shot base conserva el nombre del piloto."""
    if (prompt, k) == ("base", 0):
        return OUT / f"piloto_dev_{key}.csv"
    return OUT / f"ajuste_dev_{key}__{prompt}_k{k}.csv"


def completo(path, n):
    return path.exists() and len(pd.read_csv(path)) == n


def auc_de(path):
    r = pd.read_csv(path)
    return roc_auc_score(r.label, r.p_fake)


def ejecutar(key, tareas, inputs, comprobar=False):
    """tareas = [(csv, kwargs de llm.run)]. Carga el modelo solo si alguna tarea está pendiente."""
    pend = [(p, kw) for p, kw in tareas if not completo(p, len(inputs))]
    if not pend:
        print(f"{key}: todo hecho"); return
    if not torch.cuda.is_available():
        print(f"{key}: {len(pend)} ejecuciones pendientes pero no hay GPU; se salta"); return
    cfg = MODELOS[key]
    print(f"== {key} ({cfg['name']}): {len(pend)} ejecución(es) pendiente(s)")
    tok, model = llm.load_model(cfg["name"], cfg.get("quant"))
    if comprobar:   # comprobación a mano del formato con tres noticias
        print("Tokens de respuesta:", {k: [tok.decode([i]) for i in v] for k, v in llm.answer_ids(tok).items()})
        for i in range(3):
            p = llm.render(tok, llm.build_messages(inputs.headline[i], inputs.body[i]))
            print(f"  ejemplo {i} (label={inputs.label[i]}): {llm.generate_answer(model, tok, p)!r}")
    for p, kw in pend:
        print(f"-- {p.name}")
        llm.run(model, tok, inputs, p, **kw)
    del model, tok
    llm.release(cfg["name"], delete_cache=BORRAR_CACHE)
    if BORRAR_CACHE:
        shutil.rmtree("/tmp/hf/hub", ignore_errors=True)   # release() borra otra ruta

In [ ]:
for key in EJECUTAR:
    ejecutar(key, [(dev_path(key), {})], dev_in, comprobar=True)

## Resultados en dev

Se compara con el baseline char-SVM entrenado solo con train (mismo protocolo que en `02_baselines`). Columnas extra: `p_formato` medio (≈1 si el modelo responde «Verdadera»/«Falsa»), número de `nan` y segundos por noticia.

In [ ]:
tr, dev = splits["train"], splits["dev"]
rows = []

svm = bl.make_pipe("char", 1.0, "svm").fit(pp.prepare(tr, media_re), tr.label)
s = bl.get_scores(svm, pp.prepare(dev, media_re))
rows.append({**ev.evaluate(dev.label, ev.predict_threshold(s, 0.0), s, "char-svm (baseline)"), "decision": "umbral"})
rows.append({**ev.evaluate(dev.label, ev.predict_by_rank(s, 0.5), s, "char-svm (baseline)"), "decision": "rango 50 %"})

piloto = {}
for key in MODELOS:
    path = OUT / f"piloto_dev_{key}.csv"
    if not path.exists():
        continue
    r = piloto[key] = pd.read_csv(path)
    s, y = r.p_fake.to_numpy(), r.label.to_numpy()
    extra = {"p_formato": r.p_formato.mean(), "nan": int(r.nan.sum()), "s_noticia": r.segundos.mean()}
    rows.append({**ev.evaluate(y, ev.predict_threshold(s, 0.5), s, key), "decision": "umbral 0,5", **extra})
    rows.append({**ev.evaluate(y, ev.predict_by_rank(s, 0.5), s, key), "decision": "rango 50 %", **extra})

cols = ["modelo", "decision", "f1_fake", "precision", "recall", "pct_pred_fake", "auc", "p_formato", "nan", "s_noticia", "n"]
tabla = ev.results_table(rows)
tabla = tabla[[c for c in cols if c in tabla.columns]]
tabla.to_csv(OUT / "piloto_dev_resumen.csv", index=False)
tabla

### Sesgo de cada modelo

Distribución de `p_fake` por clase real. Si las dos clases quedan pegadas a 0 o a 1, el modelo está sesgado hacia una respuesta: el umbral 0,5 falla aunque el AUC sea bueno.

In [ ]:
for key, r in piloto.items():
    print(f"== {key}")
    display(r.groupby("label").p_fake.describe(percentiles=[.1, .5, .9]).round(3))

## Decisión del piloto (4.1)

*Resultado de la ejecución del piloto; la elección final de configuración se hace en 4.4.*

- **Modelo principal:** `gemma4-31b` (google/gemma-4-31B-it, 4 bits).
- **Segundo candidato:** `qwen38-27b` (Qwen/Qwen3.8-27B, 4 bits). Pasa también al ajuste del prompt y se compara con el principal en paralelo.
- **AUC / F1 (rango) en dev:** 0,981 / 0,930 (n = 290). Baseline char-SVM: 0,886 / 0,803.
- **Motivo:** mayor AUC y mayor F1 con rango de los cuatro modelos, con un tiempo razonable (3,2 s/noticia). Con umbral 0,5 queda equilibrado (51,7 % predicho Fake; F1 0,913).
- **Por qué se mantienen dos:** `qwen38-27b` tiene AUC 0,973, F1 rango 0,908 y **F1 con umbral 0,5 de 0,934** (superior al de gemma4-31b), a 2,8 s/noticia. La diferencia de AUC (0,008) está dentro del ruido con solo 290 noticias, así que la elección definitiva se hará tras el ajuste del prompt.
- **Regla de decisión en las siguientes fases:** el F1 con rango supone conocer la prevalencia (50 %) y sirve para comparar con los baselines. El resultado principal será el de **umbral**, que es el aplicable en un uso real.
- **Problemas encontrados:**
  - Formato y `nan`: sin incidencias (`p_formato` ≥ 0,999 y 0 `nan` en los cuatro).
  - Sesgo con umbral 0,5: `qwen3-8b` tiende a responder «Falsa» (64,8 % predicho Fake) y `gemma3-12b` a «Verdadera» (40,0 %); los de 27–31B salen equilibrados.
  - Calibración: `gemma4-31b` da probabilidades saturadas (diferencias de logit de ±27). Sirve para ordenar y decidir por rango, pero `p_fake` no es una probabilidad calibrada; si se combina con otras señales o se fija un umbral habrá que calibrarla (p. ej. Platt o isotónica sobre dev) o decidir por rango.
  - Memoria: los modelos de 27–31B solo caben en las 2×T4 con 4 bits.
  - Dev tiene solo 290 noticias: las diferencias entre los dos mejores no son concluyentes.

## 4.2 Ajuste del prompt y 4.3 few-shot

Pasan los `TOP_N` = 2 modelos con mayor AUC en el piloto (hoy `gemma4-31b` y `qwen38-27b`). Para cada uno:

1. **Prompt `detallado`** en zero-shot (4.2), comparado con el `base` del piloto.
2. **Few-shot con 4 y 8 ejemplos** (4.3), usando el mejor de los dos prompts para ese modelo. Los ejemplos salen de **train** (nunca de dev ni de test), mitad Fake y mitad Verdadera, alternados, con semilla fija y el cuerpo acortado a 150 palabras para que el prompt no se dispare. Se vigila en la tabla que el few-shot no sesgue el modelo hacia una clase (`pct_fake_umbral`).

Cada modelo se carga una sola vez para todas sus ejecuciones pendientes. Tiempo orientativo en 2×T4: 15 min por ejecución zero-shot y de 40 a 60 min por ejecución few-shot; si la sesión se corta, se reanuda.

In [ ]:
# Los TOP_N modelos con mayor AUC en el piloto pasan al ajuste
auc_piloto = {k: auc_de(dev_path(k)) for k in MODELOS if completo(dev_path(k), len(dev_in))}
elegidos = sorted(auc_piloto, key=auc_piloto.get, reverse=True)[:TOP_N]
print("AUC del piloto:", {k: round(v, 3) for k, v in auc_piloto.items()}, "-> ajuste con", elegidos)

EJEMPLOS = {k: llm.select_examples(train_in, k, SEED_EJ, MAX_WORDS_EJ) for k in K_FEWSHOT}
for k, ej in EJEMPLOS.items():
    print(f"k={k}: {[('Falsa' if y else 'Verdadera') for _, _, y in ej]}")

for key in elegidos:
    # 4.2: prompt detallado en zero-shot
    ejecutar(key, [(dev_path(key, "detallado", 0), {"system": llm.PROMPTS["detallado"]})], dev_in)
    # 4.3: few-shot con el mejor de los dos prompts para este modelo (si hay empate, el base)
    puntos = {p: auc_de(dev_path(key, p)) for p in llm.PROMPTS if completo(dev_path(key, p), len(dev_in))}
    mejor_prompt = max(puntos, key=puntos.get)
    print(f"{key}: prompt para few-shot = {mejor_prompt}  {({p: round(a, 3) for p, a in puntos.items()})}")
    ejecutar(key, [(dev_path(key, mejor_prompt, k), {"system": llm.PROMPTS[mejor_prompt], "examples": EJEMPLOS[k]})
                   for k in K_FEWSHOT], dev_in)

## 4.4 Elección de la configuración y 4.5 test

Se reúnen todas las ejecuciones completas en dev, se estima un IC 95 % del AUC por *bootstrap* (1000 remuestreos) y se elige la de mayor AUC entre las válidas (`p_formato` ≥ 0,95 y sin `nan`), desempatando por F1 con umbral 0,5. Con 290 noticias las diferencias pequeñas no son concluyentes: la tabla marca qué configuraciones tienen un AUC dentro del IC de la elegida.

La configuración elegida se guarda en `results/llm/mejor_config.json` y se puntúa **test una sola vez**, sin calcular métricas con él (eso es la tarea 6).

In [ ]:
def bootstrap_auc(y, s, n=1000, seed=42):
    rng, aucs = np.random.default_rng(seed), []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            aucs.append(roc_auc_score(y[i], s[i]))
    return np.percentile(aucs, [2.5, 97.5])


rows = []
for key in MODELOS:
    for prompt in llm.PROMPTS:
        for k in (0, *K_FEWSHOT):
            p = dev_path(key, prompt, k)
            if not completo(p, len(dev_in)):
                continue
            r = pd.read_csv(p)
            y, s = r.label.to_numpy(), r.p_fake.to_numpy()
            lo, hi = bootstrap_auc(y, s)
            u, g = ev.evaluate(y, ev.predict_threshold(s, 0.5), s), ev.evaluate(y, ev.predict_by_rank(s, 0.5), s)
            rows.append({"modelo": key, "prompt": prompt, "k": k, "auc": u["auc"], "auc_lo": lo, "auc_hi": hi,
                         "f1_umbral": u["f1_fake"], "pct_fake_umbral": u["pct_pred_fake"], "f1_rango": g["f1_fake"],
                         "p_formato": r.p_formato.mean(), "nan": int(r.nan.sum()), "s_noticia": r.segundos.mean(), "n": len(r)})

res = pd.DataFrame(rows).sort_values(["auc", "f1_umbral"], ascending=False).reset_index(drop=True)
valido = (res.p_formato >= 0.95) & (res.nan == 0)
mejor = res[valido].iloc[0]
res["empate_con_mejor"] = valido & (res.auc >= mejor.auc_lo) & (res.index != res[valido].index[0])
res.round(3).to_csv(OUT / "ajuste_dev_resumen.csv", index=False)
print(f"MEJOR: {mejor.modelo} · prompt {mejor.prompt} · k={mejor.k}  (AUC {mejor.auc:.3f}, IC95 [{mejor.auc_lo:.3f}, {mejor.auc_hi:.3f}])")
res.round(3)

In [ ]:
test_csv = None
if PUNTUAR_TEST:
    test_in = llm.llm_inputs(splits["test"], media_re)
    test_csv = OUT / f"test_{mejor.modelo}__{mejor.prompt}_k{mejor.k}.csv"
    ejecutar(mejor.modelo, [(test_csv, dict(system=llm.PROMPTS[mejor.prompt], examples=EJEMPLOS.get(mejor.k)))], test_in)

test_ok = test_csv is not None and completo(test_csv, len(splits["test"]))
config = {
    "modelo": mejor.modelo, "hf_name": MODELOS[mejor.modelo]["name"], "quant": MODELOS[mejor.modelo].get("quant"),
    "prompt": mejor.prompt, "k_ejemplos": int(mejor.k),
    "semilla_ejemplos": SEED_EJ, "max_palabras_ejemplo": MAX_WORDS_EJ, "max_palabras_cuerpo": llm.MAX_WORDS,
    "dev": {"auc": float(mejor.auc), "auc_ic95": [float(mejor.auc_lo), float(mejor.auc_hi)],
            "f1_umbral_0.5": float(mejor.f1_umbral), "pct_pred_fake_umbral": float(mejor.pct_fake_umbral),
            "f1_rango_50": float(mejor.f1_rango), "n": int(mejor.n)},
    "dev_csv": dev_path(mejor.modelo, mejor.prompt, mejor.k).relative_to(ROOT).as_posix(),
    "test_csv": test_csv.relative_to(ROOT).as_posix() if test_ok else None,
}
(OUT / "mejor_config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(config, indent=2, ensure_ascii=False))
if not test_ok:
    print("\nAVISO: test sin puntuar (sin GPU o PUNTUAR_TEST=False). Ejecutar en Kaggle antes de subir.")

## Decisión final de la tarea 4

*(Rellenar tras ejecutar en Kaggle y pasar al diario.)*

- **Configuración elegida** (la de `mejor_config.json`): modelo / prompt / nº de ejemplos:
- **AUC dev (IC 95 %) / F1 con umbral 0,5 / % predicho Fake:**
- **Frente al baseline char-SVM** (AUC 0,886, F1 0,806 en dev):
- **¿Mejoró el prompt `detallado` al base? ¿Y el few-shot al zero-shot?** (y en qué modelo):
- **Configuraciones indistinguibles de la elegida** (IC solapados) y por qué se eligió esta:
- **Problemas** (formato, `nan`, memoria, tiempo, sesgo hacia una clase):

## Entrega para las tareas 5 y 6

- `results/llm/mejor_config.json`: configuración elegida (modelo, prompt, k, semilla de los ejemplos, métricas en dev y ruta de los CSV).
- `results/llm/test_<modelo>__<prompt>_k<k>.csv`: `p_fake` de test con esa configuración. La tarea 6 lo calibra y lo evalúa sin necesidad de GPU. **No se han calculado métricas de test aquí.**
- `src/llm.py`: `PROMPTS`, `select_examples`, `build_messages` y `run` son reutilizables para el RAG (tarea 5): basta con pasar los fragmentos recuperados como `examples` o ampliar el prompt de usuario.
- Para retomar: `git pull`, ejecutar este notebook (lo ya hecho se salta) y leer `mejor_config.json`.